# J2 après-midi · 2. Un cache devant une fonction lente

Livret de TP J2 après-midi, **exercices 3, 4 et 5**.

La fonction `meilleures_cuisines(quartier)` joue le rôle d'une requête lente : elle relit les
25 359 restaurants du fichier à chaque appel. Dans une vraie application, ce serait une requête
MongoDB ou SQL coûteuse. Exécutez les deux cellules suivantes sans les modifier.

In [ ]:
import os
import redis

# Le kit donne l'adresse de Redis dans la variable REDIS_URL : redis://redis:6379/0
r = redis.from_url(os.environ["REDIS_URL"], decode_responses=True)
r.ping()

In [ ]:
import collections
import json
import time

def meilleures_cuisines(quartier):
    """Les 5 cuisines les mieux notées d'un quartier : score moyen le plus BAS
    (à New York, un score d'inspection bas est un bon score), parmi les cuisines
    inspectées au moins 100 fois. Relit les 25 359 restaurants à chaque appel : c'est lent."""
    scores = collections.defaultdict(list)
    with open("/donnees/mongo/restaurants.json", encoding="utf-8") as fichier:
        for ligne in fichier:
            resto = json.loads(ligne)
            if resto["borough"] == quartier:
                for inspection in resto["grades"]:
                    if inspection["score"] is not None:
                        scores[resto["cuisine"]].append(inspection["score"])
    moyennes = [{"cuisine": cuisine, "score_moyen": round(sum(liste) / len(liste), 2), "inspections": len(liste)}
                for cuisine, liste in scores.items() if len(liste) >= 100]
    moyennes.sort(key=lambda m: m["score_moyen"])
    return moyennes[:5]

## Exercice 3 : Mesurer la fonction lente

**Question 1.** Mesurez la durée d'un appel pour Brooklyn avec `time.perf_counter()`, qui donne
l'heure en secondes, très précisément : on le lit avant et après, on fait la différence.

**Question 2.** Lisez le résultat affiché par la même cellule. Quelle est la cuisine la mieux
notée de Brooklyn ? Attention au sens du score : à New York, un score d'inspection **bas** est
un **bon** score.

In [ ]:
# À COMPLÉTER : lire time.perf_counter() avant et après l'appel
debut = ...
resultat = meilleures_cuisines("Brooklyn")
duree = ...
print(f"{duree:.3f} s")
resultat

**Question 3.** Appelez-la trois fois de suite. Le deuxième appel est-il plus rapide ? Pourquoi ?

In [ ]:
for essai in range(1, 4):
    debut = time.perf_counter()
    meilleures_cuisines("Brooklyn")
    print(f"appel {essai} : {(time.perf_counter() - debut) * 1000:.0f} ms")

## Exercice 4 : Le cache-aside

**Question 1.** Écrivez `meilleures_cuisines_en_cache(quartier)` :

1. chercher la clé `cache:cuisines:<quartier>` dans Redis ;
2. si elle existe (succès de cache), rendre la valeur, décodée avec `json.loads` ;
3. sinon (échec de cache), appeler `meilleures_cuisines`, ranger le résultat dans Redis en JSON
   (`json.dumps`) pour 600 secondes, puis le rendre.

Redis ne range que des chaînes : la liste de dictionnaires passe en texte JSON, et inversement.

In [ ]:
DUREE_CACHE = 600   # secondes

def meilleures_cuisines_en_cache(quartier):
    cle = f"cache:cuisines:{quartier}"
    # À COMPLÉTER
    en_cache = ...
    if ...:                                       # succès de cache : on lit Redis
        return ...
    resultat = ...                                # échec de cache : on calcule
    ...                                           # on range le résultat pour DUREE_CACHE secondes
    return resultat

**Question 2.** Mesurez trois appels de suite pour Brooklyn, en partant d'un cache vide.
Combien de fois le deuxième appel est-il plus rapide que le premier ?

In [ ]:
r.delete("cache:cuisines:Brooklyn")   # on part d'un cache vide
durees = []
for essai in range(1, 4):
    # À COMPLÉTER : mesurer l'appel, ranger la durée dans durees, l'afficher en ms
    ...
print(f"le deuxième appel est {durees[0] / durees[1]:.0f} fois plus rapide")

**Question 3.** Que contient la clé dans Redis, et combien de temps lui reste-t-il ?

In [ ]:
# À COMPLÉTER : la valeur brute de la clé, puis son TTL
print(...)
...

**Question 4.** Appelez la fonction pour Manhattan, deux fois. Pourquoi le premier appel est-il
lent ?

In [ ]:
for essai in range(1, 3):
    debut = time.perf_counter()
    meilleures_cuisines_en_cache("Manhattan")
    print(f"Manhattan, appel {essai} : {(time.perf_counter() - debut) * 1000:.1f} ms")

## Exercice 5 : Invalider le cache

Une nouvelle inspection arrive pour un restaurant de Brooklyn : le résultat en cache est peut-être
faux, pendant encore 10 minutes au plus.

**Question 1.** Écrivez `invalider(quartier)`, qui supprime la clé du quartier et renvoie le
nombre de clés supprimées.

In [ ]:
def invalider(quartier):
    # À COMPLÉTER
    ...

invalider("Brooklyn")

**Question 2.** Rappelez `meilleures_cuisines_en_cache("Brooklyn")` deux fois en mesurant. Que
se passe-t-il ? Puis invalidez un quartier qui n'est pas en cache (Bronx) : que renvoie
`invalider` ?

In [ ]:
for essai in range(1, 3):
    debut = time.perf_counter()
    meilleures_cuisines_en_cache("Brooklyn")
    print(f"après invalidation, appel {essai} : {(time.perf_counter() - debut) * 1000:.1f} ms")
invalider("Bronx")